# 04 — Metrics, 4E Analysis & Scenario Economics
**APV-AD Project | Step 4 of 7**

Reads `optimal_params.json` and computes:
1. CAPEX breakdown + OpEx
2. Revenue for scenarios S0–S6
3. NPV, IRR, LCOE, payback
4. 4E: Energetic / Economic / Energo-economic / Environmental
5. Exports `results_summary.csv` — 28 rows (4 sites × 7 scenarios)

**Run time:** < 1 min | **Output:** `outputs/csv/results_summary.csv`


In [2]:
import sys, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd()))
from config_00 import SITES, PARAMS, SCENARIOS, CSV_DIR, FIG_DIR, FIG_STYLE

OPT_PATH = CSV_DIR / "optimal_params.json"
if not OPT_PATH.exists():
    raise FileNotFoundError("Run 03_optimizer.ipynb first.")
OPT = json.loads(OPT_PATH.read_text())
print("Loaded:", list(OPT.keys()))


Loaded: ['Konya', 'Almeria', 'Ouagadougou', 'Freiburg']


In [3]:
# =============================================================================
# CAPEX BREAKDOWN
# =============================================================================
def compute_capex(site, r):
    P = PARAMS
    capex_pv   = r["n_modules"] * P["P_STC_Wp"] * P["capex_pv_usd_Wp"]
    capex_ad   = r["V_dig_m3"]  * P["capex_ad_usd_m3"]
    capex_agri = P["capex_agri_usd_ha"]
    capex_bop  = P["bop_fraction"] * (capex_pv + capex_ad)
    capex_tot  = capex_pv + capex_ad + capex_agri + capex_bop
    opex_yr    = capex_tot * P["opex_fraction"]
    return dict(CAPEX_PV_kUSD=round(capex_pv/1e3,2),
                CAPEX_AD_kUSD=round(capex_ad/1e3,2),
                CAPEX_agri_kUSD=round(capex_agri/1e3,2),
                CAPEX_BOP_kUSD=round(capex_bop/1e3,2),
                CAPEX_total_kUSD=round(capex_tot/1e3,2),
                opex_kUSD_yr=round(opex_yr/1e3,2),
                _capex=capex_tot, _opex=opex_yr)

print("CAPEX preview:")
print(f"  {'Site':15s} {'PV':>8} {'AD':>6} {'Agri':>6} {'BOP':>6} {'Total':>8} {'OpEx':>8}")
for site, r in OPT.items():
    c = compute_capex(site, r)
    print(f"  {site:15s} {c['CAPEX_PV_kUSD']:8.0f} {c['CAPEX_AD_kUSD']:6.0f} "
          f"{c['CAPEX_agri_kUSD']:6.0f} {c['CAPEX_BOP_kUSD']:6.0f} "
          f"{c['CAPEX_total_kUSD']:8.0f} {c['opex_kUSD_yr']:8.0f}  k$")


CAPEX preview:
  Site                  PV     AD   Agri    BOP    Total     OpEx
  Konya                492      3      2     74      571        9  k$
  Almeria              492      3      2     74      572        9  k$
  Ouagadougou          492      3      2     74      571        9  k$
  Freiburg             492      2      2     74      571        9  k$


In [4]:
# =============================================================================
# REVENUE BY SCENARIO
# =============================================================================
def compute_revenue(site, r, scenario):
    P, cfg, sc = PARAMS, SITES[site], SCENARIOS[scenario]
    rev_pv     = r["PV_sold_MWh_ha"] * 1000 * cfg["p_elec_usd_kWh"]
    fruit_t    = cfg["fruit_yield_t_ha"] * r["LER_crop"]
    rev_tomato = fruit_t * cfg["p_tomato_usd_t"]
    E_bg_adj   = r["E_biogas_kWh_ha"] * sc["BMP_mult"] * sc["eta_cap"]
    if sc["biogas_price"] == "biomethane":
        p_bg = cfg["p_elec_usd_kWh"] * P["biomethane_premium_factor"]
    elif sc["biogas_price"] == "LPG":
        p_bg = P["p_LPG_usd_kWh"]
    else:
        p_bg = cfg["p_biogas_usd_kWh"]
    rev_biogas = E_bg_adj * p_bg
    rev_digest = 0.0
    if sc["digestate_N"]:
        rev_digest = (fruit_t*1000*cfg["R_res"]
                      * P["N_content_kg_per_kg_residue"]
                      * P["N_price_usd_kg"])
    rev_carbon = 0.0
    if sc["carbon_credit"]:
        CO2 = (r["PV_sold_MWh_ha"]*cfg["grid_ef_tCO2_MWh"]
               + E_bg_adj/1000*P["ef_natural_gas_tCO2_MWh"])
        rev_carbon = CO2 * cfg["carbon_price_usd_tCO2"]
    gross = rev_pv + rev_tomato + rev_biogas + rev_digest + rev_carbon
    return dict(rev_pv=rev_pv, rev_tomato=rev_tomato,
                rev_biogas=rev_biogas, rev_digest=rev_digest,
                rev_carbon=rev_carbon, gross=gross,
                gross_kUSD=round(gross/1e3,3))

print("S0 revenue preview:")
for site, r in OPT.items():
    rv = compute_revenue(site, r, "S0")
    print(f"  {site:15s}  PV={rv['rev_pv']/1e3:.1f}  "
          f"Tomato={rv['rev_tomato']/1e3:.1f}  "
          f"Biogas={rv['rev_biogas']/1e3:.2f}  "
          f"Total={rv['gross_kUSD']:.1f} k$/yr")


S0 revenue preview:
  Konya            PV=77.1  Tomato=6.5  Biogas=0.29  Total=84.0 k$/yr
  Almeria          PV=134.8  Tomato=8.8  Biogas=0.34  Total=143.9 k$/yr
  Ouagadougou      PV=180.4  Tomato=6.0  Biogas=0.48  Total=186.8 k$/yr
  Freiburg         PV=94.7  Tomato=3.9  Biogas=0.26  Total=98.8 k$/yr


In [5]:
# =============================================================================
# INVESTMENT METRICS
# =============================================================================
def _npv(capex, net_cf, r, N=None):
    N = N or PARAMS["project_life_yr"]
    if r == 0: return -capex + net_cf * N
    return -capex + net_cf * (1-(1+r)**(-N))/r

def _irr(capex, net_cf, N=None):
    N = N or PARAMS["project_life_yr"]
    if net_cf <= 0 or _npv(capex, net_cf, 0, N) < 0: return float("nan")
    lo, hi = 1e-6, 10.0
    for _ in range(120):
        mid = (lo+hi)/2
        if _npv(capex, net_cf, mid, N) > 0: lo = mid
        else: hi = mid
    return (lo+hi)/2

def _lcoe(capex_pv, opex_pv, E_kWh, r, N=None):
    N = N or PARAMS["project_life_yr"]
    if E_kWh <= 0 or r <= 0: return float("nan")
    ann = (1-(1+r)**(-N))/r
    return (capex_pv + opex_pv*ann) / (E_kWh*ann)

print("Investment metric functions defined. ✓")


Investment metric functions defined. ✓


In [6]:
# =============================================================================
# BUILD RESULTS_SUMMARY.CSV — 28 rows (4 sites × 7 scenarios)
# =============================================================================
rows = []

for site, r in OPT.items():
    cap  = compute_capex(site, r)
    cfg  = SITES[site]
    disc = cfg["discount_rate"]
    P    = PARAMS

    # 4E environmental (fixed per site, not per scenario)
    CO2_PV  = r["PV_sold_MWh_ha"] * cfg["grid_ef_tCO2_MWh"]
    CO2_bg  = (r["E_biogas_kWh_ha"]/1000) * P["ef_natural_gas_tCO2_MWh"]
    CO2_tot = CO2_PV + CO2_bg
    c_val   = CO2_tot * cfg["carbon_price_usd_tCO2"]

    # LCOE PV (fixed per site)
    cpv = cap["CAPEX_PV_kUSD"]*1e3
    pv_opex_share = cpv / cap["_capex"]
    lcoe_val = _lcoe(cpv, cap["_opex"]*pv_opex_share,
                     r["PV_sold_MWh_ha"]*1000, disc)

    # Biogas cost (fixed per site)
    cad = cap["CAPEX_AD_kUSD"]*1e3
    ad_share = cad / cap["_capex"]
    ann  = (1-(1+disc)**(-P["project_life_yr"]))/disc if disc>0 else P["project_life_yr"]
    cbg  = ((cad + cap["_opex"]*ad_share*ann)
            / (r["E_biogas_kWh_ha"]*ann)
            if r["E_biogas_kWh_ha"]>0 else float("nan"))

    for sc_name in SCENARIOS:
        rv    = compute_revenue(site, r, sc_name)
        net   = rv["gross"] - cap["_opex"]
        NPV   = _npv(cap["_capex"], net, disc)
        IRR   = _irr(cap["_capex"], net)
        PB    = cap["_capex"]/net if net > 0 else float("inf")

        row = {
            "site": site, "scenario": sc_name,
            # Design
            "beta_deg": round(r["beta_deg"],2),
            "d_row_m":  round(r["d_row_m"],3),
            "H_m_m":    round(r["H_m_m"],3),
            "V_dig_m3": round(r["V_dig_m3"],2),
            "HRT_days": round(r["HRT_days"],1),
            "f_PV_heat":round(r["f_PV_heat"],4),
            "GCR_pct":  round(r["GCR_pct"],2),
            "OLR_kgVS_m3d": round(r["OLR_kgVS_m3d"],3),
            # E1
            "PV_total_MWh_ha":     round(r["PV_total_MWh_ha"],1),
            "PV_sold_MWh_ha":      round(r["PV_sold_MWh_ha"],1),
            "PV_self_MWh_ha":      round(r["PV_total_MWh_ha"]-r["PV_sold_MWh_ha"],1),
            "biogas_total_MWh_ha": round(r["biogas_total_MWh_ha"],2),
            "heat_demand_MWh_ha":  round(r["heat_demand_MWh_ha"],2),
            "biogas_for_heat_MWh_ha": 0.0,
            "ESR":                 round(r["ESR"],1),
            "BMP_avg_NmL_gVS":     round(r["BMP_avg_NmL_gVS"],1),
            # eLER
            "LER_crop":    round(r["LER_crop"],3),
            "LER_PV_defA": round(r["LER_PV_defA"],3),
            "LER_PV_defB": round(r["LER_PV_defB"],3),
            "LER_biogas":  round(r["LER_biogas"],3),
            "eLER_defA":   round(r["eLER_defA"],3),
            "eLER_defB":   round(r["LER_crop"]+r["LER_PV_defB"]+r["LER_biogas"],3),
            "LER_2C":      round(r["LER_2C"],3),
            # Water (corrected)
            "W_saved_mm_season":    round(r["W_saved_mm_season"],1),
            "ET0_season_mm":        round(r["ET0_season_mm"],1),
            "ET0_annual_mm":        round(r["ET0_annual_mm"],1),
            "LER_water_season_pct": round(r["LER_water_season_pct"],2),
            "LER_water_annual_pct": round(r["LER_water_annual_pct"],2),
            # CAPEX
            "CAPEX_PV_kUSD":    cap["CAPEX_PV_kUSD"],
            "CAPEX_AD_kUSD":    cap["CAPEX_AD_kUSD"],
            "CAPEX_agri_kUSD":  cap["CAPEX_agri_kUSD"],
            "CAPEX_BOP_kUSD":   cap["CAPEX_BOP_kUSD"],
            "CAPEX_total_kUSD": cap["CAPEX_total_kUSD"],
            # E2
            "gross_revenue_kUSD_yr": round(rv["gross"]/1e3,2),
            "opex_kUSD_yr":          round(cap["_opex"]/1e3,2),
            "net_CF_kUSD_yr":        round(net/1e3,2),
            "NPV_kUSD":    round(NPV/1e3,1),
            "IRR_pct":     round(IRR*100,2) if IRR==IRR else None,
            "payback_yr":  round(PB,1) if PB < 1000 else None,
            # E3
            "LCOE_PV_USD_kWh":    round(lcoe_val,4),
            "cost_biogas_USD_MWh":round(cbg,2) if cbg==cbg else None,
            # E4
            "CO2_avoided_PV_tCO2_ha":     round(CO2_PV,1),
            "CO2_avoided_biogas_tCO2_ha":  round(CO2_bg,2),
            "CO2_avoided_total_tCO2_ha":   round(CO2_tot,1),
            "carbon_value_USD_ha":         round(c_val,1),
            # Validation
            "V1_MAPE_pct": None,
            "V2_MAPE_pct": None,
        }
        rows.append(row)

df = pd.DataFrame(rows)
out = CSV_DIR / "results_summary.csv"
df.to_csv(out, index=False, float_format="%.4f")
print(f"Saved → {out}")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")


Saved → C:\Users\AMIDOU MAIGA\OneDrive - Institut 2IE\Desktop\APV + AD\APV_AD_Project\outputs\csv\results_summary.csv
Shape: 28 rows × 49 columns


In [7]:
# =============================================================================
# SUMMARY PIVOTS
# =============================================================================
print("\neLER (Def A) — S0 baseline:")
s0 = df[df["scenario"]=="S0"].set_index("site")
for site in s0.index:
    r = s0.loc[site]
    print(f"  {site:15s} eLER={r['eLER_defA']:.3f}  "
          f"LER_2C={r['LER_2C']:.3f}  "
          f"NPV={r['NPV_kUSD']:.0f}k$  "
          f"IRR={r['IRR_pct']:.1f}%  "
          f"PB={r['payback_yr']:.1f}yr")

print("\nNPV (k$) — all scenarios:")
piv = df.pivot(index="scenario", columns="site", values="NPV_kUSD")
print(piv.to_string(float_format=lambda x: f"{x:8.0f}"))

print("\nIRR (%) — all scenarios:")
piv2 = df.pivot(index="scenario", columns="site", values="IRR_pct")
print(piv2.to_string(float_format=lambda x: f"{x:7.1f}"))

print("\nCO2 avoided total (tCO2/ha) — S0:")
for site in s0.index:
    print(f"  {site:15s} {s0.loc[site,'CO2_avoided_total_tCO2_ha']:.0f} tCO2/ha")

print("\nLCOE PV (USD/kWh) — S0:")
for site in s0.index:
    print(f"  {site:15s} {s0.loc[site,'LCOE_PV_USD_kWh']:.3f} USD/kWh")

print("\n" + "="*55)
print("  04_metrics — COMPLETE")
print("="*55)
print("  Share results_summary.csv for discussion analysis.")
print("  Next step → 05_figures.ipynb")



eLER (Def A) — S0 baseline:
  Konya           eLER=1.653  LER_2C=1.038  NPV=169k$  IRR=11.8%  PB=7.6yr
  Almeria         eLER=1.659  LER_2C=1.042  NPV=981k$  IRR=23.3%  PB=4.2yr
  Ouagadougou     eLER=1.725  LER_2C=1.089  NPV=947k$  IRR=31.1%  PB=3.2yr
  Freiburg        eLER=1.758  LER_2C=1.031  NPV=554k$  IRR=14.8%  PB=6.3yr

NPV (k$) — all scenarios:
site      Almeria  Freiburg    Konya  Ouagadougou
scenario                                         
S0            981       554      169          947
S1            981       554      170          947
S2            980       553      169          946
S3            985       557      173          952
S4            982       554      170          947
S5            999       568      178          976
S6           1168       878      252         1044

IRR (%) — all scenarios:
site      Almeria  Freiburg   Konya  Ouagadougou
scenario                                        
S0           23.3      14.8    11.8         31.1
S1           23.3    